# **06. PREDICTIVE MODEL**

## **Greater London house price forecast**
La notebook se encarga de documentar el proceso de creación y evaluación de un modelo predictivo, `Prophet`, para generar un pronóstico de la evolución de las viviendas en el área metropolitana de Londres.

### **Objetivo**
Realizar una predicción de la serie temporal a 5 años, mostrando la `evolución del precio medio anual` de la vivienda en la zona y el intervalo de confianza con el que ha predicho el modelo.

### **Tabla de entrada**
`workspace.default.london_gold` — 2,384,979 filas · 22 columnas


In [0]:
%pip install prophet

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

## **1. Visualización de los datos: Capa GOLD**

Antes de proceder a la creacion del modelo predcitivo, se visualizan los datos en su version mas optima, el dataset de la Capa GOLD de transformacóon. De aquí, el modelo se encargará de generar sus pronósticos.

In [0]:
df_pred = spark.table("workspace.default.london_gold")
display(df_pred.limit(5))

## **2. Agregación y preparación de datos en PySpark**

Se agregan los datos a nivel mensual y se calcula el precio medio. Una vez agregados, se procede a transformar el df a pandas para el correcto funcionamiento del modelo. 

Posteriormente, se separan los datos en TRAIN y TEST para validar el funciomaiento correcto del modelo y que no exista sobreajuste. Se reservan los ultimos 12 meses para probar el modelo.

In [0]:
df_monthly_spark = df_pred.groupBy("year", "month") \
                          .agg(F.mean("price").alias("avg_price")) \
                          .orderBy("year", "month")

df_pandas = df_monthly_spark.toPandas()
df_pandas['ds'] = pd.to_datetime(df_pandas[['year', 'month']].assign(day=1))
df_prophet = df_pandas[['ds', 'avg_price']].rename(columns={'avg_price': 'y'})

test_months = 12
df_train = df_prophet.iloc[:-test_months]
df_test = df_prophet.iloc[-test_months:]

print("Agregación en PySpark y conversión a Pandas completada.")
print(f"Datos de entrenamiento (Train): {len(df_train)} meses listos para el modelo.")
print(f"Datos de validación (Test): {len(df_test)} meses reservados para medir el error.")

## **3. Entrenamiento del modelo y cálculo del error (MAE y RMSE)**

Se ha seleccionado `Prophet` como algoritmo principal por su robustez al modelar series temporales con fuerte estacionalidad, permitiendo generar predicciones a 5 años con intervalos de confianza claros y altamente interpretables.

En primer lugar, se instancia el modelo con la estacionalidad deseada para los objetivos. Se desactivan las estacionalidades semanales al ser datos mensuales agregados. Posteriormente, se procede a entrenar el modelo y a validar sus resultados:

In [0]:
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.015
)

model.fit(df_train)
print("Modelo entrenado con los datos históricos (Train).")

future_test = df_test[['ds']]
forecast_test = model.predict(future_test)

y_true = df_test['y'].values
y_pred = forecast_test['yhat'].values

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print("Resultados de la Validación (Últimos 12 meses):")
print(f"   - MAE  : {mae}")
print(f"   - RMSE : {rmse}")

comparativa = df_test[['ds', 'y']].rename(columns={'y': 'y_true'})
comparativa['y_pred'] = y_pred
comparativa['error_absoluto'] = abs(comparativa['y_true'] - comparativa['y_pred'])

display(comparativa)

Se observa que el modelo tiene un error medio absoluto `(MAE)` de `79698.81` y una raiz del error cuadratico medio `(RMSE)` de `93045.66`. Esto indica que el modelo se equivoca en un promedio de 79.698 GBP prediciendo los precios de la vivienda. El RMSE ligeramente mas alto indica erroes algo mayores el algunos meses puntuales. 

## **4. Visualización de la predicción**

Una vez entrenado y validado el modelo, se procede a visualizar la predicción para tener una imagen clara de la dirección de los precios de la vivienda según las predicciones generadas.

In [0]:
modelo_final = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.015 
)

modelo_final.fit(df_prophet) 
print("Modelo final entrenado con el 100% de los datos históricos.")

futuro = modelo_final.make_future_dataframe(periods=60, freq='MS')
prediccion = modelo_final.predict(futuro)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df_prophet['ds'], df_prophet['y'], 
        color='#1B2A4A', label='Precio histórico', linewidth=1.5)

solo_futuro = prediccion[prediccion['ds'] > df_prophet['ds'].max()]

ax.plot(solo_futuro['ds'], solo_futuro['yhat'], 
        color='#E8A020', linestyle='--', label='Predicción 2025-2029', linewidth=2)
ax.fill_between(solo_futuro['ds'], 
                solo_futuro['yhat_lower'], solo_futuro['yhat_upper'], 
                alpha=0.2, color='#E8A020', label='Intervalo de confianza')
ax.set_title('Greater London House Prices — Historical & Forecast (5 Years)')
ax.set_xlabel('Año')
ax.set_ylabel('Precio Medio (£)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Se observa el crecimiento constante del valor de las viviendas en Greater London durante estos 5 siguientes años a un ritmo constante. La última caída de precio en 2025 no es representativa en cuanto al crecimiento general de los inmuebles. 

## **Resumen de la predicción**

A continuación, se muestra una tabla con los valores medios predichos por año y los intervalos de confianza de cada predicción:

In [0]:

prediccion['año'] = prediccion['ds'].dt.year
resumen_anual = prediccion.groupby('año')[['yhat', 'yhat_lower', 'yhat_upper']].mean().reset_index()
año_maximo_historico = df_prophet['ds'].dt.year.max()
resumen_futuro = resumen_anual[resumen_anual['año'] > año_maximo_historico]
resumen_futuro = resumen_futuro.round(2)
display(resumen_futuro)

Como visto en la gráfica, el valor de la vivienda sube gradualmente durante estos años siguientes, llegando el precio medio a un maximo de `752.728 GBP` para el año 2030. Este valor tiene un intervalo de confianza desde `716133 GBP hasta 789018 GBP`. 